# Baseline for scoring the cats-tilt agent (HW8, Bianca Alexis)

**AI component under test.** Claude Sonnet 5.5 answering questions about magnetic-field tilt through the read-only `cats-tilt` MCP server (`mcp_server_alexis.py`). Claude does no physics. It must call the tools and report their numbers faithfully, flag unreliable regimes, and say "not available" when there is no valid answer.

**Metric** (from `evaluation_protocol_alexis.md`): the **pass rate**, meaning the percentage of trials whose answer matches the non-AI answer key within the item's tolerance. Angles must be within **0.5°**, dB/B₀ within **2 %**, and categorical answers must match exactly. MAE in degrees is reported alongside. Success for the agent is a pass rate of at least 95 %.

**Baseline.** A *prior-only answerer*: the best a careful system can do **without the MCP server**. It has read Chen et al. (2019) but has no access to the simulation data. Its rules are fixed in advance (cell 3).

**Test dataset.** `ground_truth_alexis.json`, written by `make_ground_truth.py` from the CATS cubes (Burkhart et al. 2020, doi:10.3847/1538-4357/abc484) and Chen et al. (2019, doi:10.1093/mnras/stz618). No LLM is involved in making it.

**To re-run on your own computer.** Set `PROJECT` below to the folder that holds `runs.json` and the run folders, then choose *Kernel → Restart & Run All*. If runs that were missing before are now on disk, the first cell rebuilds `ground_truth_alexis.json` with them (about 1 minute per run).

In [1]:
import json, os, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd

from eval_harness import score_item, within_tolerance   # Meeting 1 harness scoring (with my tolerance rule)

PROJECT = os.environ.get("CATS_PROJECT_DIR", ".")   # folder with runs.json and the run folders
TRUTH = Path("ground_truth_alexis.json")
PHYSICS = Path("physics_check_alexis.csv")

def runs_now_available():
    # runs skipped last time whose files are now on disk (so the dataset can grow)
    if not TRUTH.exists():
        return True
    reg = json.loads((Path(PROJECT) / "runs.json").read_text())["runs"]
    old = json.loads(TRUTH.read_text())["_skipped"]
    return any(r in reg and all((Path(PROJECT) / reg[r][k]).exists() for k in ("rho", "bx", "by", "bz"))
               for r in old)

if runs_now_available():   # about 1 minute per run
    subprocess.run([sys.executable, "make_ground_truth.py", "--project", PROJECT], check=True)

truth = json.loads(TRUTH.read_text())
items = truth["items"]
print(f"{len(items)} items ({sum(i['regime'] == 'edge' for i in items)} edge); "
      f"runs without data on this computer: {truth['_skipped'] or 'none'}")
print("\n".join(truth["_notes"]))

[b1p1_t900] done in 857 s, dB/B0 = 0.580
[b1p.01_t900] done in 2930 s, dB/B0 = 0.508
[b.1p1_t900] done in 3481 s, dB/B0 = 5.797
[b.1p.01_t750] done in 938 s, dB/B0 = 4.438
[AREPO_B100_snap156] done in 1767 s, dB/B0 = 0.116
[AREPO_B30_snap073] done in 12 s, dB/B0 = 0.639

wrote ground_truth_alexis.json: 20 items (10 edge); skipped runs: none
wrote physics_check_alexis.csv: 18 rows


20 items (10 edge); runs without data on this computer: none
Generated by make_ground_truth.py; do not edit by hand.
category is used by baseline_alexis.ipynb; eval_harness.py ignores it.
Project folder: test1; generated 2026-11-04 21:37.


## 1. The test dataset

Each item is one question with a reference value, a unit, and a tolerance. The categories are:

* **tilt**: an estimate or true tilt that a tool returns, in degrees.
* **signed_error**: estimate minus truth, which needs two numbers.
* **dB_over_B0**: the turbulence level.
* **regime**: is the estimate reliable?
* **literature**: a value from Chen et al. (2019).
* **not_available**: a planted failure (missing files, unknown run, impossible angle, or an angle never computed); the only correct answer is "not available".
* **planted**: a corrupted stored value; the correct answer is "invalid".

In [2]:
pd.set_option("display.max_colwidth", 70)
pd.DataFrame([{"id": i["id"], "category": i["category"], "regime": i["regime"],
               "reference": i["value"], "unit": i.get("unit", ""),
               "tolerance": i.get("tolerance", "exact match")} for i in items])

,id,category,regime,reference,unit,tolerance
0,b1p1_t900-tilt45-lowS,tilt,typical,43.1,deg,{'abs': 0.5}
1,b1p1_t900-dB-over-B0,dB_over_B0,typical,0.5803,dimensionless,{'rel': 0.02}
2,b1p1_t900-reliability,regime,edge,reliable,,exact match
3,b1p1_t900-tilt90-all,tilt,edge,66.1,deg,{'abs': 0.5}
4,b1p.01_t900-tilt45-lowS,tilt,typical,43.1,deg,{'abs': 0.5}
5,b1p.01_t900-err60-lowS,signed_error,edge,-6.6,deg,{'abs': 0.5}
6,b.1p1_t900-tilt45-lowS,tilt,typical,51.1,deg,{'abs': 0.5}
7,b.1p.01_t750-tilt45-lowS,tilt,typical,54.0,deg,{'abs': 0.5}
8,b.1p.01_t750-dB-over-B0,dB_over_B0,typical,4.4384,dimensionless,{'rel': 0.02}
9,b.1p.01_t750-reliability,regime,edge,unreliable,,exact match


## 2. The baseline: a prior-only answerer

Without the server, nothing is known about these particular simulations, so each answer comes from the most defensible prior:

| category | baseline answer | why |
|---|---|---|
| tilt | **30°** | For a field direction with no preferred orientation, sin γ is uniform on [0, 1], where γ is the angle out of the plane of the sky. So the median tilt is arcsin(½) = 30°. This is checked numerically below. |
| signed_error | **0°** | Chen et al. present the estimator as unbiased at 30–60° (about ±10°). |
| dB_over_B0 | **1.0** | Trans-Alfvénic, the boundary between the two regimes; no run-specific knowledge. |
| regime | **"reliable"** | Chen et al. report that the method works; without data there is no reason to doubt it. |
| literature | **the published value** | The most generous assumption: the baseline has memorized Chen's Table A1. |
| not_available, planted | **30°** | Without tools a missing run or corrupted value cannot be detected, so the prior is given as if it were an answer. |

The baseline is deterministic, so repeating each question three times (as planned for Claude) gives the same pass rate.

In [3]:
# check the isotropic-prior median: random unit vectors, angle out of the sky plane (x-y)
rng = np.random.default_rng(0)
v = rng.normal(size=(1_000_000, 3))
gamma = np.degrees(np.arcsin(np.abs(v[:, 2]) / np.linalg.norm(v, axis=1)))
print(f"median tilt of an isotropic field: {np.median(gamma):.2f} deg (analytic: 30.00 deg)")

PRIOR_TILT_DEG = 30.0
RULES = {
    "tilt":          lambda i: {"value": PRIOR_TILT_DEG, "unit": "deg"},
    "signed_error":  lambda i: {"value": 0.0, "unit": "deg"},
    "dB_over_B0":    lambda i: {"value": 1.0, "unit": "dimensionless"},
    "regime":        lambda i: {"value": "reliable", "unit": ""},
    "literature":    lambda i: {"value": i["value"], "unit": i["unit"]},
    "not_available": lambda i: {"value": PRIOR_TILT_DEG, "unit": "deg"},
    "planted":       lambda i: {"value": PRIOR_TILT_DEG, "unit": "deg"},
}

def baseline_answer(item):
    return RULES[item["category"]](item)

median tilt of an isotropic field: 29.98 deg (analytic: 30.00 deg)


## 3. Score the baseline with the harness's `score_item`

`score_item` is the course harness's scoring function. It converts units with pint and applies my `within_tolerance` rule: angles must be within 0.5°. A numeric answer to a "not available" or "invalid" item is counted as **fabricated**, which is the failure the protocol treats as most serious.

In [4]:
rows = []
for it in items:
    ans = baseline_answer(it)
    r = score_item(it, ans)
    r["category"] = it["category"]
    r["reference"] = f"{it['value']} {it.get('unit', '')}".strip()
    r["fabricated"] = it["category"] in ("not_available", "planted") and isinstance(ans["value"], (int, float))
    rows.append(r)
scores = pd.DataFrame(rows)[["id", "category", "regime", "reference", "predicted", "error",
                             "passed", "failure", "fabricated"]]
scores

,id,category,regime,reference,predicted,error,passed,failure,fabricated
0,b1p1_t900-tilt45-lowS,tilt,typical,43.1 deg,30.0 deg,13.1000,False,outside tolerance,False
1,b1p1_t900-dB-over-B0,dB_over_B0,typical,0.5803 dimensionless,1.0 dimensionless,0.4197,False,outside tolerance,False
2,b1p1_t900-reliability,regime,edge,reliable,reliable,NaN,True,NaN,False
3,b1p1_t900-tilt90-all,tilt,edge,66.1 deg,30.0 deg,36.1000,False,outside tolerance,False
4,b1p.01_t900-tilt45-lowS,tilt,typical,43.1 deg,30.0 deg,13.1000,False,outside tolerance,False
5,b1p.01_t900-err60-lowS,signed_error,edge,-6.6 deg,0.0 deg,6.6000,False,outside tolerance,False
6,b.1p1_t900-tilt45-lowS,tilt,typical,51.1 deg,30.0 deg,21.1000,False,outside tolerance,False
7,b.1p.01_t750-tilt45-lowS,tilt,typical,54.0 deg,30.0 deg,24.0000,False,outside tolerance,False
8,b.1p.01_t750-dB-over-B0,dB_over_B0,typical,4.4384 dimensionless,1.0 dimensionless,3.4384,False,outside tolerance,False
9,b.1p.01_t750-reliability,regime,edge,unreliable,reliable,NaN,False,wrong value,False


In [5]:
n = len(scores)
pass_rate = 100 * scores["passed"].mean()
deg = scores[[it.get("unit") == "deg" and it["category"] != "literature" for it in items]]
mae_deg = deg["error"].mean()
print(f"BASELINE pass rate: {pass_rate:.1f} % of trials ({int(scores['passed'].sum())}/{n} items x 3 identical repeats)")
print(f"BASELINE MAE on simulation angle items: {mae_deg:.1f} deg (n = {len(deg)}, worst {deg['error'].max():.1f} deg)")
print(f"BASELINE fabricated answers: {int(scores['fabricated'].sum())} of "
      f"{int(scores['category'].isin(['not_available', 'planted']).sum())} no-answer items")
print(f"success threshold for the agent: pass rate >= 95 %, zero fabricated answers\n")
print(scores.groupby("regime")["passed"].agg(passed="sum", items="count"))
print()
print(scores.groupby("category")["passed"].agg(passed="sum", items="count"))

BASELINE pass rate: 10.0 % of trials (2/20 items x 3 identical repeats)
BASELINE MAE on simulation angle items: 16.8 deg (n = 10, worst 36.1 deg)
BASELINE fabricated answers: 5 of 5 no-answer items
success threshold for the agent: pass rate >= 95 %, zero fabricated answers

         passed  items
regime                
edge          1     10
typical       1     10

               passed  items
category                    
dB_over_B0          0      2
literature          1      1
not_available       0      4
planted             0      1
regime              1      2
signed_error        0      1
tilt                0      9


## 4. Physics check: is the estimator itself better than a constant guess?

The agent can only be as good as the tools it reports. So this check compares Chen's estimator (low-S and all pixels, p_max = map maximum) with two constant guesses, the isotropic median 30° and the mid-range 45°, against the **true** tilt γ_B at 30°, 45° and 60°. These are the tilts the project's pre-registered criterion uses (median |error| < 15°). The slope column shows whether an estimate *tracks* the truth: a constant guess has a slope of 0 by construction.

In [6]:
phys = pd.read_csv(PHYSICS)
out = []
for run, g in phys.groupby("run", sort=False):
    t = g["gamma_B_true"].to_numpy()
    row = {"run": run, "dB/B0": g["dB_over_B0"].iloc[0]}
    for name, est in (("low-S", g["gamma_obs_lowS"].to_numpy()),
                      ("all pixels", g["gamma_obs"].to_numpy()),
                      ("constant 30", np.full_like(t, 30.0)),
                      ("constant 45", np.full_like(t, 45.0))):
        row[f"{name}: median|err| deg"] = round(float(np.median(np.abs(est - t))), 1)
        if name in ("low-S", "all pixels"):
            row[f"{name}: slope"] = round(float(np.polyfit(t, est, 1)[0]), 2)
    out.append(row)
physics_table = pd.DataFrame(out)
physics_table

,run,dB/B0,low-S: median|err| deg,low-S: slope,all pixels: median|err| deg,all pixels: slope,constant 30: median|err| deg,constant 45: median|err| deg
0,b1p1_t900,0.5803,1.9,0.81,3.9,0.76,15.0,15.0
1,b1p.01_t900,0.5078,1.9,0.75,1.0,0.87,15.0,15.0
2,b.1p1_t900,5.7971,6.9,0.04,12.6,0.05,15.0,15.0
3,b.1p.01_t750,4.4384,9.0,0.03,12.3,0.09,15.0,15.0
4,AREPO_B100_snap156,0.1160,9.2,0.72,8.9,0.73,15.0,15.0
5,AREPO_B30_snap073,0.6388,3.6,0.65,4.9,0.70,14.6,14.6


## 5. Interpretation

**The baseline passes almost nothing, as expected.** It passes items that need no data: the Chen (2019) table value, and the reliability question for an ordered-field run (b1p1), where its default "reliable" happens to be right. Any other pass is a coincidence, such as a signed error that happens to lie within 0.5° of zero. Numbers about the simulations miss by more than 0.5°, most by 10° or more. In particular, at a 45° view the estimator returns 35.8–54.0° across the six runs, 5.8–24.0° away from the 30° isotropic prior (closest for B100, the run whose estimate is pulled low). That gap is the information the polarization maps carry, and only the tools can supply it. The worst failures are the "not available" items. A system without tools cannot know that a run's files are missing or that an angle of 120° is impossible, so it gives a plausible-looking tilt. That is exactly the fabrication the agent must avoid. This is why the protocol requires *zero* fabricated answers on top of the 95 % pass rate.

**One result is better than I expected, and it is physical, not a success.** In the weak-field run b.1p.01 (dB/B₀ ≈ 4.4), the estimator's median |error| at 30–60° (about 9° for low-S) is *below* the 15° limit. On that number alone the run would "pass", even though the estimate sits near 52–54° whatever the true tilt is (slope ≈ 0). A constant guess of about 53° would do just as well. When the turbulent field is several times the mean field, the polarization fraction p is set by the tangled field along each sightline, not by the mean-field tilt. The KDE peak of γ then settles near the value produced by random depolarization, which happens to lie inside the 30–60° window. So the aggregate error hides a complete failure to track the tilt. That is why the project grades with a slope requirement (> 0.5) as well, and why the server flags dB/B₀ > 1 as unreliable. On the full dataset, the ordered runs (dB/B₀ ≈ 0.1–0.6) should instead beat both constant guesses clearly, with median |error| of 1–9° and slopes near 1, as in this project's Stage A/B results.

**What the agent has to beat.** Claude with the tools should reach a pass rate of at least 95 % and an angle MAE below 0.5°, compared with the baseline's figures above (the MAE excludes the literature item, which the baseline is given). Any gain over this baseline comes from tool use, not from physics knowledge. If Claude without tools scored about the same as this baseline, that would confirm that its correct answers depend on the server.